# Civil Engineering Project Copilot — PRODUCTION MIRROR End to End

> **PRODUCTION MIRROR:** this notebook imports and runs the same application code used by the API and UI. It does not reimplement retrieval, tools, agents, memory, tracing, or evaluation.

## Contract: one inspectable control surface

This walkthrough covers offline indexing and online questioning. **No silent fallback** is allowed: the requested `portable`, `local`, or `live` mode either starts as requested or raises a clear error. Portable is the safe default and needs no Docker, network, or credentials.

## Architecture at a glance

![End-to-end Civil Copilot architecture](../docs/images/civil-copilot-architecture-overview.png)

*The overview separates preparation from question-time retrieval, tools, and agents.*

The notebook follows the same four responsibilities shown in the proposal: data ingestion, data retrieval, agents, and typed read-only tools.

## What we will demonstrate

- **Example A:** an exact RFI question through Hybrid retrieval and Fast RAG.
- **Example B:** a connected delay investigation through Graph RAG and Bounded ReAct.
- Preference memory, grounded citations, abstention, structured traces, and Evaluation are shown with the same production objects.

In [1]:
import inspect
import json
import os
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

from IPython.display import Markdown, display

from civil_copilot.agents.react import ReactAgentSuite
from civil_copilot.agents.state import ChatRequest
from civil_copilot.agents.tool_registry import DEFAULT_TOOL_REGISTRY
from civil_copilot.config import Settings
from civil_copilot.data.loaders import load_corpus
from civil_copilot.data.synthetic import default_gold_scenarios
from civil_copilot.evals.runner import EvaluationRunner
from civil_copilot.ingestion.service import IngestionService
from civil_copilot.memory.service import PreferenceMemory
from civil_copilot.retrieval.answer import GroundedAnswerService
from civil_copilot.retrieval.evidence import EvidencePacket, RetrievalTrace
from civil_copilot.retrieval.query import QueryContext
from civil_copilot.runtime import (
    RuntimeMode,
    build_application_runtime,
    build_runtime,
)

## Execution modes

- **portable** — deterministic in-process stores, deterministic embeddings, a scripted tool-calling model, in-memory preference memory, and tracing disabled; no services or keys.
- **local** — PostgreSQL, Qdrant, and Neo4j on configured local endpoints; local uses deterministic embeddings.
- **live** — the same external stores, but live uses OpenAI embeddings.

In this notebook, local and live require an OpenAI key unless a model is supplied. The public factory can accept a supplied model; this walkthrough does not provide one. Local and live use Mem0 when its key is configured, otherwise safe preference memory stays in process. Local and live enable Langfuse when both keys are configured.

Set `COPILOT_NOTEBOOK_MODE` before launching Jupyter. This notebook never starts containers and never prints secrets.

In [2]:
requested_mode = RuntimeMode(os.getenv("COPILOT_NOTEBOOK_MODE", "portable").strip().lower())
runtime_settings = (
    Settings(
        _env_file=None,
        openai_api_key=None,
        mem0_api_key=None,
        langfuse_public_key=None,
        langfuse_secret_key=None,
    )
    if requested_mode is RuntimeMode.PORTABLE
    else Settings()
)
if requested_mode is RuntimeMode.PORTABLE:
    assert runtime_settings.openai_api_key is None
    assert runtime_settings.mem0_api_key is None
    assert runtime_settings.langfuse_public_key is None
    assert runtime_settings.langfuse_secret_key is None
print(f"requested_mode={requested_mode.value}")
print("Portable mode ignores credentials; no secret values will be displayed.")

requested_mode=portable
Portable mode ignores credentials; no secret values will be displayed.


In [3]:
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path.cwd().parent
corpus = load_corpus(ROOT)
scenarios = default_gold_scenarios()
print(f"records={len(corpus.records)}, chunks={len(corpus.chunks)}, relationships={len(corpus.relationships)}")
print(f"gold_scenarios={len(scenarios)}")

records=333, chunks=383, relationships=460
gold_scenarios=7


## Data sources and provenance

Every item keeps its origin label and source location. **SYNTHETIC — ACADEMIC DEMO** records create the connected construction story. **PUBLIC** records are official public previews or catalogue material; they are not represented as complete Indian Standards.

In [4]:
origin_counts = {}
for record in corpus.records:
    origin_counts[record.data_origin] = origin_counts.get(record.data_origin, 0) + 1
pprint(origin_counts)
public_examples = [
    {"record_id": item.record_id, "origin": item.data_origin, "source": item.source_path}
    for item in corpus.records
    if item.data_origin == "public_official"
][:3]
pprint(public_examples)

{'public_official': 88, 'synthetic_academic_demo': 245}
[{'origin': 'public_official',
  'record_id': 'PUBLIC-BIS-bis-10262',
  'source': 'data/public/bis/academic/INDEX.jsonl#bis-10262'},
 {'origin': 'public_official',
  'record_id': 'PUBLIC-BIS-bis-12843-1989-reff2021',
  'source': 'data/public/bis/academic/INDEX.jsonl#bis-12843-1989-reff2021'},
 {'origin': 'public_official',
  'record_id': 'PUBLIC-BIS-bis-1343-2012-amd1-reff2022',
  'source': 'data/public/bis/academic/INDEX.jsonl#bis-1343-2012-amd1-reff2022'}]


In [5]:
scenario_by_id = {item.scenario_id: item for item in scenarios}
example_a = scenario_by_id["S-01"]
example_b = scenario_by_id["S-04"]
standards_example = scenario_by_id["S-07"]
graph_subquestion = scenario_by_id["S-03"]
for item in (example_a, graph_subquestion, example_b):
    print(f"{item.scenario_id} | {item.expected_route} | {item.question}")

S-01 | rag | What did RFI-087 decide, and which drawing revision contains the decision?
S-03 | graph_rag | What is downstream of RFI-087 if its decision is not implemented?
S-04 | agentic_rag | Why was activity ACT-STEEL-009 blocked, what changed, and what evidence closes the issue?


## Offline indexing

![Data ingestion architecture](../docs/images/data-ingestion-architecture.png)

*Source records are checked, split into searchable passages, and published with their origin labels.*

“Indexing” means validating the connected corpus and publishing the same records to the record, search, and graph stores. Portable starts with empty in-process stores for this demonstration. Local and live preserve existing records in their explicitly requested persistent stores, then create, update, or retain each incoming item. Calling the production ingestion service again is restart-safe: unchanged items are counted instead of duplicated.

In [6]:
previous_application = globals().get("application")
if previous_application is not None:
    previous_application.close()
application = build_application_runtime(
    mode=requested_mode,
    settings=runtime_settings,
    corpus=corpus,
    initialize_data=False,
)
assert isinstance(application.react_agents, ReactAgentSuite)
assert isinstance(application.memory, PreferenceMemory)
assert isinstance(application.evaluator, EvaluationRunner)
if requested_mode is RuntimeMode.PORTABLE:
    print("Portable runtime assembled with empty in-process stores.")
else:
    print("External runtime assembled; existing persistent data was preserved.")

Portable runtime assembled with empty in-process stores.


In [7]:
capabilities = application.capabilities.model_dump(mode="json")
integration_status = {
    "chat_model": "deterministic portable" if requested_mode is RuntimeMode.PORTABLE else "configured OpenAI",
    "embeddings": "OpenAI" if requested_mode is RuntimeMode.LIVE else "deterministic",
    "memory_backend": type(application.memory.backend).__name__,
    "tracing_enabled": application.tracing.enabled,
}
pprint(capabilities)
pprint(integration_status)
print(f"fallback_allowed={capabilities['fallback_allowed']}")
assert capabilities["mode"] == requested_mode.value
assert capabilities["fallback_allowed"] is False

{'checkpoint_backend': 'memory',
 'durable_checkpoints': False,
 'fallback_allowed': False,
 'graph_backend': 'networkx',
 'mode': 'portable',
 'records_backend': 'memory',
 'search_backend': 'memory_bm25_and_deterministic_dense',
 'server_filtered': False}
{'chat_model': 'deterministic portable',
 'embeddings': 'deterministic',
 'memory_backend': 'InMemoryPreferenceBackend',
 'tracing_enabled': False}
fallback_allowed=False


In [8]:
first_publish = application.ingestion.ingest(
    corpus.records,
    corpus.chunks,
    corpus.relationships,
)
first_publish_data = asdict(first_publish)
pprint(first_publish_data)
print("FIRST_PUBLISH " + json.dumps(first_publish_data, sort_keys=True))
if requested_mode is RuntimeMode.PORTABLE:
    assert first_publish.records.created > 0
    assert first_publish.chunks.created > 0
    assert first_publish.graph_nodes.created > 0
    assert first_publish.relationships.created > 0
else:
    assert first_publish.records.created + first_publish.records.updated + first_publish.records.unchanged == len(corpus.records)
    assert first_publish.chunks.created + first_publish.chunks.updated + first_publish.chunks.unchanged == len(corpus.chunks)
    assert first_publish.graph_nodes.created + first_publish.graph_nodes.updated + first_publish.graph_nodes.unchanged == len(corpus.records)
    assert first_publish.relationships.created + first_publish.relationships.updated + first_publish.relationships.unchanged == len(corpus.relationships)

{'chunks': {'created': 383, 'unchanged': 0, 'updated': 0},
 'graph_nodes': {'created': 333, 'unchanged': 0, 'updated': 0},
 'records': {'created': 333, 'unchanged': 0, 'updated': 0},
 'relationships': {'created': 460, 'unchanged': 0, 'updated': 0}}
FIRST_PUBLISH {"chunks": {"created": 383, "unchanged": 0, "updated": 0}, "graph_nodes": {"created": 333, "unchanged": 0, "updated": 0}, "records": {"created": 333, "unchanged": 0, "updated": 0}, "relationships": {"created": 460, "unchanged": 0, "updated": 0}}


In [9]:
second_publish = application.ingestion.ingest(
    corpus.records,
    corpus.chunks,
    corpus.relationships,
)
second_publish_data = asdict(second_publish)
pprint(second_publish_data)
print("SECOND_PUBLISH " + json.dumps(second_publish_data, sort_keys=True))
assert second_publish.records.unchanged == len(corpus.records)
assert second_publish.chunks.unchanged == len(corpus.chunks)
assert second_publish.graph_nodes.unchanged == len(corpus.records)
assert second_publish.relationships.unchanged == len(corpus.relationships)

{'chunks': {'created': 0, 'unchanged': 383, 'updated': 0},
 'graph_nodes': {'created': 0, 'unchanged': 333, 'updated': 0},
 'records': {'created': 0, 'unchanged': 333, 'updated': 0},
 'relationships': {'created': 0, 'unchanged': 460, 'updated': 0}}
SECOND_PUBLISH {"chunks": {"created": 0, "unchanged": 383, "updated": 0}, "graph_nodes": {"created": 0, "unchanged": 333, "updated": 0}, "records": {"created": 0, "unchanged": 333, "updated": 0}, "relationships": {"created": 0, "unchanged": 460, "updated": 0}}


### What changes by mode?

The calls above stay the same. Portable writes into newly empty in-process production store adapters on each runtime construction. Local/live publish to existing PostgreSQL, Qdrant, and Neo4j stores without clearing shared data; the first report can therefore contain created, updated, and unchanged items. A local/live connection error is surfaced—it is never replaced with portable results.

In [10]:
print("records_backend:", capabilities["records_backend"])
print("search_backend:", capabilities["search_backend"])
print("graph_backend:", capabilities["graph_backend"])
if requested_mode in {RuntimeMode.LOCAL, RuntimeMode.LIVE}:
    assert capabilities["records_backend"] == "postgresql"
    assert capabilities["graph_backend"] == "neo4j"

records_backend: memory
search_backend: memory_bm25_and_deterministic_dense
graph_backend: networkx


## Inspect the production source, not a notebook copy

Source links: [application runtime](../src/civil_copilot/runtime.py), [ingestion](../src/civil_copilot/ingestion/service.py), [retrieval](../src/civil_copilot/retrieval/hybrid.py), [ReAct suite](../src/civil_copilot/agents/react.py).

Set `SHOW_SOURCE=1` only when you want the notebook to display selected production source.

In [11]:
SHOW_SOURCE = os.getenv("SHOW_SOURCE", "0") == "1"
if SHOW_SOURCE:
    print(inspect.getsource(build_application_runtime))
    print(inspect.getsource(build_runtime))
    print(inspect.getsource(IngestionService.ingest))
else:
    print("Source display is off; use the links above or set SHOW_SOURCE=1.")

Source display is off; use the links above or set SHOW_SOURCE=1.


## Hybrid retrieval

![Data retrieval architecture](../docs/images/data-retrieval-architecture.png)

*Exact, word-based, meaning-based, and graph search build a small evidence set before answering.*

Hybrid retrieval combines exact identifiers, ordinary word matching (sparse/full-text), and meaning-based matching (dense vectors). Rank fusion combines those lists; reranking then moves evidence that best fits the question to the top.

In [12]:
query_a = QueryContext(
    question=example_a.question,
    project_id="BLR-STEEL-DEMO",
    access_scopes=["project:blr-steel-demo", "public"],
    top_k=6,
)
print(query_a.question)

What did RFI-087 decide, and which drawing revision contains the decision?


In [13]:
packet_a = application.retrieval.retrieve(query_a)
pprint(packet_a.retrieval_trace.model_dump())
assert packet_a.evidence
assert packet_a.evidence[0].chunk.record_id == "RFI-087"
assert packet_a.retrieval_trace.keyword_candidates > 0
assert packet_a.retrieval_trace.vector_candidates > 0

{'exact_identifiers': ['RFI-087'],
 'filtered_candidates': 373,
 'fused_candidates': 373,
 'hybrid_ranking': ['RFI-087',
                    'PUBLIC-BIS-bis-1786',
                    'DRAW-S-208-R3',
                    'DRAW-S-205-R3',
                    'DRAW-S-210-R3',
                    'DRAW-S-202-R3',
                    'PUBLIC-BIS-bis-808-2021',
                    'DRAW-S-201-R3',
                    'DRAW-S-208-R5',
                    'DRAW-S-205-R5',
                    'DRAW-S-204-R3',
                    'PUBLIC-BIS-bis-3764-1992-reff2022',
                    'PUBLIC-BIS-bis-1893-4-2024',
                    'DRAW-S-203-R3',
                    'DRAW-S-201-R5',
                    'MIN-STEEL-05',
                    'PUBLIC-BIS-bis-10262',
                    'DRAW-S-212-R3',
                    'DRAW-S-206-R3',
                    'DRAW-S-202-R5',
                    'DRAW-S-203-R5',
                    'PUBLIC-BIS-bis-516-1-1',
                    'DRAW-S-204-R5',
 

In [14]:
retrieval_rows = [
    {
        "record_id": item.chunk.record_id,
        "origin": item.chunk.data_origin,
        "fused_score": round(item.fused_score, 5),
        "rerank_score": round(item.rerank_score, 5),
        "exact": item.exact_id_match,
        "reasons": item.reasons,
    }
    for item in packet_a.evidence
]
pprint(retrieval_rows)

[{'exact': True,
  'fused_score': 0.04534,
  'origin': 'synthetic_academic_demo',
  'reasons': ['exact rank 1',
              'text rank 2',
              'dense rank 18',
              'exact record identifier',
              'question term overlap',
              'current or accepted status'],
  'record_id': 'RFI-087',
  'rerank_score': 2.29534},
 {'exact': False,
  'fused_score': 0.01639,
  'origin': 'synthetic_academic_demo',
  'reasons': ['text rank 1',
              'dense rank 82',
              'mentions RFI-087',
              'question term overlap'],
  'record_id': 'MIN-STEEL-07',
  'rerank_score': 0.36639},
 {'exact': False,
  'fused_score': 0.02516,
  'origin': 'synthetic_academic_demo',
  'reasons': ['text rank 19',
              'dense rank 20',
              'question term overlap',
              'current or accepted status'],
  'record_id': 'DRAW-S-208-R5',
  'rerank_score': 0.22516},
 {'exact': False,
  'fused_score': 0.02479,
  'origin': 'synthetic_academic_demo',
  

In [15]:
store_candidates = application.stores.search.search_hybrid(
    query=example_a.question,
    project_id="BLR-STEEL-DEMO",
    access_scopes=["project:blr-steel-demo", "public"],
    metadata_filters={},
    limit=6,
)
pprint([
    {
        "record_id": item.chunk.record_id,
        "exact_rank": item.exact_rank,
        "text_rank": item.text_rank,
        "dense_rank": item.dense_rank,
        "fused_score": round(item.fused_score, 5),
    }
    for item in store_candidates
])
assert any(item.exact_rank is not None for item in store_candidates)
assert any(item.text_rank is not None for item in store_candidates)
assert any(item.dense_rank is not None for item in store_candidates)

[{'dense_rank': 18,
  'exact_rank': 1,
  'fused_score': 0.04534,
  'record_id': 'RFI-087',
  'text_rank': 2},
 {'dense_rank': 9,
  'exact_rank': None,
  'fused_score': 0.0264,
  'record_id': 'PUBLIC-BIS-bis-1786',
  'text_rank': 24},
 {'dense_rank': 1,
  'exact_rank': None,
  'fused_score': 0.02583,
  'record_id': 'DRAW-S-208-R3',
  'text_rank': 46},
 {'dense_rank': 3,
  'exact_rank': None,
  'fused_score': 0.02558,
  'record_id': 'DRAW-S-205-R3',
  'text_rank': 43},
 {'dense_rank': 2,
  'exact_rank': None,
  'fused_score': 0.02539,
  'record_id': 'DRAW-S-210-R3',
  'text_rank': 48},
 {'dense_rank': 5,
  'exact_rank': None,
  'fused_score': 0.02538,
  'record_id': 'DRAW-S-202-R3',
  'text_rank': 40},
 {'dense_rank': 12,
  'exact_rank': None,
  'fused_score': 0.02538,
  'record_id': 'PUBLIC-BIS-bis-808-2021',
  'text_rank': 27},
 {'dense_rank': 6,
  'exact_rank': None,
  'fused_score': 0.02525,
  'record_id': 'DRAW-S-201-R3',
  'text_rank': 39},
 {'dense_rank': 20,
  'exact_rank': None,

## Fast RAG

Fast RAG means one focused retrieval-and-answer path. It is appropriate for Example A because the question names `RFI-087`. The answer service may only state claims backed by the returned evidence.

In [16]:
fast_response = application.workflow.invoke(
    ChatRequest(
        question=example_a.question,
        route_override="rag",
        user_id="notebook-reviewer",
    )
)
print("route:", fast_response.route)
print("grounded:", fast_response.grounded, "abstained:", fast_response.abstained)
display(Markdown(fast_response.answer))

route:

 rag
grounded: True abstained: False


SYNTHETIC — ACADEMIC DEMO: Structural clarification 087. Record RFI-087; type rfi; status closed; revision response-1; effective 2026-03-12. SYNTHETIC — ACADEMIC DEMO. Site requested clarification of beam-to-column connection C17 shown on S-204 Rev 3. The approved response required plate PL-17B and was incorporated in S-204 Rev 5. Activity ACT-STEEL-009… [RFI-087](http://127.0.0.1:8011/api/records/RFI-087)

In [17]:
pprint([
    {
        "record_id": citation.record_id,
        "origin": citation.data_origin,
        "source_path": citation.source_path,
        "source_url": citation.source_url,
    }
    for citation in fast_response.citations
])
assert fast_response.grounded and fast_response.citations

[{'origin': 'synthetic_academic_demo',
  'record_id': 'RFI-087',
  'source_path': 'data/synthetic/steel_building_demo/records.jsonl#RFI-087',
  'source_url': None}]


In [18]:
abstention = GroundedAnswerService().answer(
    EvidencePacket(
        question="What is the permitted source-backed answer?",
        evidence=[],
        retrieval_trace=RetrievalTrace(returned_evidence=0),
    )
)
print(abstention.answer)
assert abstention.abstained and abstention.grounded

I do not have enough evidence in the permitted project sources to answer this question.


## Graph RAG

Graph RAG follows verified project relationships—such as an RFI changing a drawing and affecting an activity—then retrieves evidence for that path. It is useful when the answer depends on connections rather than one passage.

In [19]:
graph_response = application.workflow.invoke(
    ChatRequest(
        question=graph_subquestion.question,
        route_override="graph_rag",
        user_id="notebook-reviewer",
    )
)
print("route:", graph_response.route, "paths:", len(graph_response.graph_paths))
display(Markdown(graph_response.answer))

route: graph_rag paths: 9


SYNTHETIC — ACADEMIC DEMO: Structural clarification 087. Record RFI-087; type rfi; status closed; revision response-1; effective 2026-03-12. SYNTHETIC — ACADEMIC DEMO. Site requested clarification of beam-to-column connection C17 shown on S-204 Rev 3. The approved response required plate PL-17B and was incorporated in S-204 Rev 5. Activity ACT-STEEL-009… [RFI-087](http://127.0.0.1:8011/api/records/RFI-087)

SYNTHETIC — ACADEMIC DEMO: Steel work activity 009. Record ACT-STEEL-009; type schedule_activity; status in_progress; revision baseline-2; effective 2026-02-18. SYNTHETIC — ACADEMIC DEMO. Fabricate, deliver, or erect structural steel for level 2, zone 3. Planned duration: 7 days. [ACT-STEEL-009](http://127.0.0.1:8011/api/records/ACT-STEEL-009)

SYNTHETIC — ACADEMIC DEMO: S-204 framing plan revision 5. Record DRAW-S-204-R5; type drawing; status current; revision 5; effective 2026-02-28. SYNTHETIC — ACADEMIC DEMO. Current issued-for-construction plan for grid 4, incorporating reviewed design decisions and connection details. [DRAW-S-204-R5](http://127.0.0.1:8011/api/records/DRAW-S-204-R5)

In [20]:
path_rows = [
    {
        "depth": path.depth,
        "nodes": [node.record_id for node in path.nodes],
        "relationships": [edge.relationship_type for edge in path.edges],
    }
    for path in graph_response.graph_paths[:8]
]
pprint(path_rows)
assert any("RFI-087" in row["nodes"] for row in path_rows)

[{'depth': 1,
  'nodes': ['RFI-087', 'ACT-STEEL-009'],
  'relationships': ['AFFECTS']},
 {'depth': 1,
  'nodes': ['RFI-087', 'DRAW-S-204-R5'],
  'relationships': ['CHANGES_OR_CLARIFIES']},
 {'depth': 1,
  'nodes': ['RFI-087', 'DRAW-S-204-R3'],
  'relationships': ['REFERENCES']},
 {'depth': 2,
  'nodes': ['RFI-087', 'ACT-STEEL-009', 'PROJECT-BLR-01'],
  'relationships': ['AFFECTS', 'DELIVERS']},
 {'depth': 2,
  'nodes': ['RFI-087', 'ACT-STEEL-009', 'ACT-STEEL-008'],
  'relationships': ['AFFECTS', 'DEPENDS_ON']},
 {'depth': 2,
  'nodes': ['RFI-087', 'DRAW-S-204-R5', 'DRAW-S-204-R3'],
  'relationships': ['CHANGES_OR_CLARIFIES', 'REVISES']},
 {'depth': 3,
  'nodes': ['RFI-087', 'ACT-STEEL-009', 'PROJECT-BLR-01', 'CODE-REGISTER-001'],
  'relationships': ['AFFECTS', 'DELIVERS', 'HAS_REGISTER']},
 {'depth': 3,
  'nodes': ['RFI-087', 'ACT-STEEL-009', 'ACT-STEEL-008', 'PROJECT-BLR-01'],
  'relationships': ['AFFECTS', 'DEPENDS_ON', 'DELIVERS']}]


In [21]:
expected_graph_backend = "networkx" if requested_mode is RuntimeMode.PORTABLE else "neo4j"
print("verified graph backend:", capabilities["graph_backend"])
assert capabilities["graph_backend"] == expected_graph_backend
print("Graph results came from the requested backend; no fallback occurred.")

verified graph backend: networkx
Graph results came from the requested backend; no fallback occurred.


## Preference memory

Preference memory remembers safe display choices—not RFIs, project facts, or generated answers. Portable uses in-memory preference storage. Local and live use Mem0 when its key is configured; without that key they use the same safe in-memory boundary.

In [22]:
application.memory.add(
    "notebook-reviewer",
    "BLR-STEEL-DEMO",
    "answer_style",
    "plain_language",
)
preferences = application.memory.get("notebook-reviewer", "BLR-STEEL-DEMO")
pprint(preferences)
assert preferences["answer_style"] == "plain_language"

{'answer_style': 'plain_language'}


## Registered typed tools

![Typed tools architecture](../docs/images/tools-architecture.png)

*Each tool performs one controlled read-only job and returns sources with its result.*

A typed tool accepts a small validated input and returns a structured read-only observation. The central registry controls which tools the orchestrator, Document specialist, Schedule specialist, and Risk specialist may receive. The `assess_standard_evidence` tool compares project records with the indexed IS 800 public preview in one bounded action; it does not certify compliance.

In [23]:
assert application.tool_registry is DEFAULT_TOOL_REGISTRY
tool_rows = [
    {
        "name": name,
        "owner": application.tool_registry.get(name).owning_specialist,
        "allowed_agents": application.tool_registry.get(name).allowed_agents,
        "schema": application.tool_registry.get(name).input_schema.__name__,
        "read_only": application.tool_registry.get(name).read_only,
    }
    for name in application.tool_registry.names()
]
pprint(tool_rows)

[{'allowed_agents': ('orchestrator', 'document', 'risk'),
  'name': 'search_documents',
  'owner': 'document',
  'read_only': True,
  'schema': 'SearchDocumentsInput'},
 {'allowed_agents': ('orchestrator', 'document', 'schedule', 'risk'),
  'name': 'get_record',
  'owner': 'document',
  'read_only': True,
  'schema': 'GetRecordInput'},
 {'allowed_agents': ('orchestrator', 'schedule', 'risk'),
  'name': 'query_project_graph',
  'owner': 'risk',
  'read_only': True,
  'schema': 'GraphQueryInput'},
 {'allowed_agents': ('orchestrator', 'schedule', 'risk'),
  'name': 'analyze_schedule',
  'owner': 'schedule',
  'read_only': True,
  'schema': 'ScheduleAnalysisInput'},
 {'allowed_agents': ('orchestrator', 'document'),
  'name': 'compare_revisions',
  'owner': 'document',
  'read_only': True,
  'schema': 'CompareRevisionsInput'},
 {'allowed_agents': ('orchestrator', 'schedule', 'risk'),
  'name': 'calculate',
  'owner': 'schedule',
  'read_only': True,
  'schema': 'CalculateInput'},
 {'allowed

## Bounded ReAct

![Agent orchestration architecture](../docs/images/agent-orchestration-architecture.png)

*The orchestrator chooses a specialist and the next tool after inspecting the latest observation.*

Bounded ReAct means: make a short plan, call one permitted tool, inspect its observation, then decide the next action. Limits prevent endless loops. The displayed trace is a structured execution summary—not hidden chain-of-thought.

In [24]:
tool_context = application.tool_context(
    user_id="notebook-reviewer",
    project_id="BLR-STEEL-DEMO",
    access_scopes=("project:blr-steel-demo", "public"),
)
react_result = application.run_react(
    role="orchestrator",
    question=example_b.question,
    context=tool_context,
)
agent_response = application.workflow.invoke(
    ChatRequest(
        question=example_b.question,
        route_override="agentic_rag",
        user_id="notebook-reviewer",
    )
)
print("ReAct stop reason:", react_result.stop_reason)
print("workflow route:", agent_response.route)

ReAct stop reason: completed
workflow route: agentic_rag


In [25]:
pprint([event.model_dump(mode="json") for event in react_result.trace])
pprint([
    {
        "tool": item.tool_name,
        "status": item.status,
        "summary": item.summary,
        "source_ids": item.source_ids,
    }
    for item in react_result.observations
])
assert {event.phase for event in react_result.trace} >= {"plan", "act", "observe", "decide"}
assert react_result.source_ids

[{'model_turn': 0,
  'phase': 'plan',
  'source_ids': [],
  'summary': 'Select one permitted evidence step and inspect its observation.',
  'title': 'Bounded investigation plan',
  'tool_call_id': None,
  'tool_metadata': {},
  'tool_name': None},
 {'model_turn': 1,
  'phase': 'act',
  'source_ids': [],
  'summary': 'Executing one typed read-only operation.',
  'title': 'Call compare_revisions',
  'tool_call_id': 'portable-1-compare_revisions',
  'tool_metadata': {'acl_policy': 'revision:read',
                    'allowed_agents': ['orchestrator', 'document'],
                    'description': 'Compare two permitted controlled revisions '
                                   'of one document.',
                    'input_schema': {'properties': {'document_id': {'maxLength': 160,
                                                                    'minLength': 2,
                                                                    'title': 'Document '
                                     

In [26]:
for role in ("document", "schedule", "risk", "orchestrator"):
    print(role, "->", sorted(application.react_agents.tool_names(role)))
print("Agent answer grounded source IDs:", react_result.source_ids)
print("Workflow citations:", [item.record_id for item in agent_response.citations])
assert agent_response.grounded
standards_response = application.workflow.invoke(
    ChatRequest(question=standards_example.question, user_id="notebook-reviewer")
)
standards_tools = [
    event.title for event in standards_response.trace if event.stage == "tool"
]
assert standards_tools == ["assess_standard_evidence"]
assert standards_response.grounded and not standards_response.abstained
assert "Not evidenced" in standards_response.answer
assert "not the full Indian Standard" in standards_response.answer
display(Markdown(standards_response.answer))
print("Standards citations:", [item.record_id for item in standards_response.citations])

document -> ['assess_standard_evidence', 'compare_revisions', 'get_record', 'search_documents']
schedule -> ['analyze_schedule', 'calculate', 'get_record', 'query_project_graph']
risk -> ['analyze_schedule', 'assess_standard_evidence', 'calculate', 'get_record', 'query_project_graph', 'search_documents']
orchestrator -> ['analyze_schedule', 'assess_standard_evidence', 'calculate', 'compare_revisions', 'get_record', 'query_project_graph', 'search_documents']
Agent answer grounded source IDs: ['DRAW-S-204-R3', 'DRAW-S-204-R5', 'ACT-STEEL-009', 'RFI-087', 'RFI-085', 'RFI-088', 'RFI-086', 'RFI-089']
Workflow citations: ['ACT-STEEL-009', 'DRAW-S-204-R3', 'DRAW-S-204-R5']


IS 800 evidence review for project BLR-STEEL-DEMO

**Evidenced**

- The project identifies general hot-rolled structural-steel construction as its scope. The project code entry and steel specifications identify IS 800 and structural-steel work. [Project: CODE-IS-800, SPEC-STEEL-01, SPEC-STEEL-09; BIS preview: bis-800-chunk-0001]

- Structural-steel material is identified and traceable to an Indian material standard. The project register references IS 2062 and a mill certificate records the grade and heat number. [Project: CODE-IS-2062, MTC-01-01; BIS preview: bis-800-chunk-0008]

- Welding work records a procedure, qualified welder, and inspection result. The project references an Indian welding practice and records a WPS, welder, and inspection. [Project: CODE-IS-816, WELD-001, INSP-WELD-001; BIS preview: bis-800-chunk-0002]

- Fabrication and erection work is covered by specifications and scheduled activities. A steel specification covers fabrication and erection, and the schedule records that work. [Project: SPEC-STEEL-01, ACT-STEEL-001; BIS preview: bis-800-chunk-0001]

**Needs review**

- Inspection and acceptance records are complete for the reviewed steel work. Inspection and repair records exist, but open NCR-005 means the reviewed set is not fully closed. [Project: INSP-WELD-001, NCR-005; BIS preview: bis-800-chunk-0007]

- The project load basis is demonstrated in enough detail for an engineering check. The register and calculation summary mention IS 875 and approved loads, but the indexed records do not show complete load combinations. [Project: CODE-IS-875-2, CODE-IS-875-3, CALC-FRAME-03; BIS preview: bis-800-chunk-0001]

**Not evidenced**

- Detailed seismic design evidence is present for the steel frame. The project register references IS 1893, but the available summary does not provide a detailed seismic design check. [Project: CODE-IS-1893-1, CALC-FRAME-06; BIS preview: bis-800-chunk-0001]

**Important limit:** This review compares project records only with the indexed official BIS public preview. The preview is not the full Indian Standard and cannot prove full compliance. Missing evidence is not proof that a practice was not followed; it identifies information for a qualified engineer to review.

Standards citations: ['CODE-IS-800', 'SPEC-STEEL-01', 'SPEC-STEEL-09', 'PUBLIC-BIS-bis-800', 'CODE-IS-2062', 'MTC-01-01', 'PUBLIC-BIS-bis-800', 'CODE-IS-816', 'WELD-001', 'INSP-WELD-001', 'PUBLIC-BIS-bis-800', 'ACT-STEEL-001', 'NCR-005', 'PUBLIC-BIS-bis-800', 'CODE-IS-875-2', 'CODE-IS-875-3', 'CALC-FRAME-03', 'CODE-IS-1893-1', 'CALC-FRAME-06']


## Tracing and Langfuse

The production runtime attaches a real run-specific reference to the ReAct invocation. Without Langfuse it is a local application trace ID with no web URL. Local and live enable Langfuse only when both Langfuse keys are configured; then the same public reference carries the real Langfuse trace ID and URL. The structured ReAct execution summary remains a separate teaching view. The notebook never manufactures a reference, reaches into the Langfuse client, or prints keys.

In [27]:
react_trace_reference = application.trace_reference(react_result)
assert react_trace_reference.trace_id
trace_reference_data = react_trace_reference.model_dump(mode="json")
print("TRACE_REFERENCE " + json.dumps(trace_reference_data, sort_keys=True))
if react_trace_reference.url:
    display(Markdown(f"[Open the ReAct Langfuse trace]({react_trace_reference.url})"))
else:
    print("ReAct application trace is local; no web URL is available.")
for label, result in (("fast_rag", fast_response), ("graph_rag", graph_response), ("agentic_workflow", agent_response)):
    reference = getattr(result, "trace_reference", None) or application.trace_reference(result)
    if reference.trace_id:
        print(label, "trace_id=", reference.trace_id)
        if reference.url:
            display(Markdown(f"[{label} Langfuse trace]({reference.url})"))
    else:
        print(label, "has no run-specific application trace reference")
print("Structured execution summary is separate from this application trace.")

TRACE_REFERENCE {"provider": "local", "trace_id": "local-run-4cf64ab6-ae18-4d58-92fb-73ac732f107d", "url": null}
ReAct application trace is local; no web URL is available.
fast_rag trace_id= local-run-aad0c7ea-2427-47df-99f8-83a225bb5f53
graph_rag trace_id= local-run-693c7896-00f6-49d7-b816-1068fb57c512
agentic_workflow trace_id= local-run-105bf0e7-71dc-417f-90ed-653acdabe822
Structured execution summary is separate from this application trace.


## Evaluation

Evaluation turns the demonstration into repeatable checks. Deterministic portable runs are suitable for regression tests. Local/live modes run the same gold questions against their explicitly requested services and model.

In [28]:
evaluation_scenarios = [example_a, graph_subquestion, example_b]
evaluation_report = application.evaluator.run(evaluation_scenarios)
evaluation_by_id = {item.scenario_id: item for item in evaluation_report.scenarios}
pprint(evaluation_report.aggregate)
pprint([
    {
        "scenario": item.scenario_id,
        "route": item.route,
        "recall_at_6": item.recall_at_6,
        "citation_coverage": item.citation_coverage,
        "passed": item.passed,
    }
    for item in evaluation_report.scenarios
])
route_eval_matrix = {
    "rag": {
        "question": example_a.question,
        "route": fast_response.route,
        "grounded": fast_response.grounded,
        "citation_ids": [item.record_id for item in fast_response.citations],
        "citation_coverage": evaluation_by_id[example_a.scenario_id].citation_coverage,
        "evaluation_passed": evaluation_by_id[example_a.scenario_id].passed,
    },
    "graph_rag": {
        "question": graph_subquestion.question,
        "route": graph_response.route,
        "grounded": graph_response.grounded,
        "citation_ids": [item.record_id for item in graph_response.citations],
        "citation_coverage": evaluation_by_id[graph_subquestion.scenario_id].citation_coverage,
        "evaluation_passed": evaluation_by_id[graph_subquestion.scenario_id].passed,
    },
    "agentic_rag": {
        "question": example_b.question,
        "route": agent_response.route,
        "grounded": agent_response.grounded,
        "citation_ids": [item.record_id for item in agent_response.citations],
        "citation_coverage": evaluation_by_id[example_b.scenario_id].citation_coverage,
        "evaluation_passed": evaluation_by_id[example_b.scenario_id].passed,
    },
}
assert all(item["route"] == route for route, item in route_eval_matrix.items())
assert all(item["grounded"] and item["citation_ids"] for item in route_eval_matrix.values())
assert all(item["evaluation_passed"] for item in route_eval_matrix.values())
print("ROUTE_EVAL_MATRIX " + json.dumps(route_eval_matrix, sort_keys=True))

{

'abstention_accuracy': 1.0,
 'citation_coverage': 1.0,
 'hybrid_ndcg_at_6': 0.7019870549185293,
 'ndcg_at_6': 0.7445574272354877,
 'recall_at_6': 0.8055555555555555,
 'reciprocal_rank': 0.7777777777777778,
 'reranked_ndcg_at_6': 0.7019870549185293,
 'reranker_lift_at_6': 0.0,
 'route_accuracy': 1.0,
 'scenario_pass_rate': 1.0,
 'tool_selection_precision': 0.7333333333333334,
 'unnecessary_step_rate': 0.13333333333333333}
[{'citation_coverage': 1.0,
  'passed': True,
  'recall_at_6': 1.0,
  'route': 'rag',
  'scenario': 'S-01'},
 {'citation_coverage': 1.0,
  'passed': True,
  'recall_at_6': 0.6666666666666666,
  'route': 'graph_rag',
  'scenario': 'S-03'},
 {'citation_coverage': 1.0,
  'passed': True,
  'recall_at_6': 0.75,
  'route': 'agentic_rag',
  'scenario': 'S-04'}]
ROUTE_EVAL_MATRIX {"agentic_rag": {"citation_coverage": 1.0, "citation_ids": ["ACT-STEEL-009", "DRAW-S-204-R3", "DRAW-S-204-R5"], "evaluation_passed": true, "grounded": true, "question": "Why was activity ACT-STEEL-009

In [29]:
evaluation_kind = "deterministic portable regression" if requested_mode is RuntimeMode.PORTABLE else "explicit local/live evaluation"
print("evaluation kind:", evaluation_kind)
print("scenario pass rate:", evaluation_report.aggregate["scenario_pass_rate"])
assert evaluation_report.scenario_count == len(evaluation_scenarios)
if requested_mode is RuntimeMode.LIVE:
    assert application.tracing.enabled
    print("Live evaluation completed with configured model and tracing.")

evaluation kind: deterministic portable regression
scenario pass rate: 1.0


## Why this notebook will not diverge

It contains orchestration calls and displays only—no production business function or class definitions. Changing the runtime, retriever, stores, registered tools, ReAct workflow, memory, tracing, or evaluator changes what this notebook executes automatically.

More source links: [workflow](../src/civil_copilot/agents/workflow.py), [tools](../src/civil_copilot/agents/tools.py), [memory](../src/civil_copilot/memory/service.py), [evaluation](../src/civil_copilot/evals/runner.py).

In [30]:
assert application.capabilities.fallback_allowed is False
assert packet_a.evidence[0].chunk.record_id in {item.record_id for item in fast_response.citations}
assert graph_response.graph_paths
assert react_result.observations
assert evaluation_report.scenario_count == 3
application.close()
print(f"PRODUCTION_MIRROR_OK requested_mode={requested_mode.value} fallback_allowed=False")

PRODUCTION_MIRROR_OK requested_mode=portable fallback_allowed=False
